In [1]:
import asyncio
import json
import os
from typing import Annotated, Any, Never

from agent_framework import (
    AgentExecutor,
    AgentExecutorRequest,
    AgentExecutorResponse,
    ChatMessage,
    Role,
    WorkflowBuilder,
    WorkflowContext,
    ai_function,
    executor,
)

# 🤖 GitHub Models or OpenAI client integration
from agent_framework.openai import OpenAIChatClient
from dotenv import load_dotenv
from IPython.display import HTML, display
from pydantic import BaseModel

print("✅ All imports successful!")

✅ All imports successful!


## Step 1: నిర్మిత అవుట్‌పుట్స్ కోసం Pydantic మోడెళ్లను నిర్వచించండి

ఈ మోడెళ్లే ఏజెంట్లు తిరిగి ఇచ్చే **స్కీమా**ని నిర్వచిస్తాయి. Pydanticతో `response_format` ఉపయోగించడం ద్వారా ఈ గుణాలు ఉంటాయి:
- ✅ టైప్-సేఫ్ డేటా ఎక్స్‌ప్ట్రాక్షన్
- ✅ ఆటోమేటిక్ వాలిడేషన్
- ✅ ఫ్రీ-టెక్స్ట్ రెస్పాన్స్‌ల నుండి పార్సింగ్ లోపాలు ఉండదీ
- ✅ ఫీల్డ్లపై ఆధారపడి సులభమైన కండిషనల్ రౌటింగ్


In [2]:
class BookingCheckResult(BaseModel):
    """Result from checking hotel availability at a destination."""

    destination: str
    has_availability: bool
    message: str


class AlternativeResult(BaseModel):
    """Suggested alternative destination when no rooms available."""

    alternative_destination: str
    reason: str


class BookingConfirmation(BaseModel):
    """Booking suggestion when rooms are available."""

    destination: str
    action: str
    message: str


print("✅ Pydantic models defined:")
print("   - BookingCheckResult (availability check)")
print("   - AlternativeResult (alternative suggestion)")
print("   - BookingConfirmation (booking confirmation)")

✅ Pydantic models defined:
   - BookingCheckResult (availability check)
   - AlternativeResult (alternative suggestion)
   - BookingConfirmation (booking confirmation)


## Step 2: హోటల్ బుకింగ్ టూల్ తయారు చేయండి

ఈ టూల్ అనేది **availability_agent** కాల్ చేసి గదులు అందుబాటులో ఉన్నాయా లేకడా తనిఖీ చేయడానికి ఉపయోగిస్తుంది. మేము `@ai_function` డెకరేటర్ ఉపయోగించాము:
- Python ఫంక్షన్‌ను AI-కాల్ చేయదగిన టూల్‌గా మలచడానికి
- LLM కోసం JSON స్కీమాను స్వయంచాలకంగా రూపొందించడానికి
- పరామితి ధ్రువీకరణ నిర్వహించడానికి
- ఏజెంట్ల ద్వారా ఆటోమేటిక్ కాల్‌ను అనుమతించడానికి

ఈ డెమో కోసం:
- **స్టాక్‌హోమ్, సియాటిల్, టోక్యో, లండన్, ఆమన్స్టర్డామ్** → గదులు ఉన్నాయి ✅
- **ఇతర అన్ని నగరాలు** → గదులు లేవు ❌


In [3]:
@ai_function(description="Check hotel room availability for a destination city")
def hotel_booking(destination: Annotated[str, "The destination city to check for hotel rooms"]) -> str:
    """
    Simulates checking hotel room availability.
    
    Returns JSON string with availability status.
    """
    display(
        HTML(f"""
        <div style='padding: 15px; background: #e3f2fd; border-left: 4px solid #2196f3; border-radius: 4px; margin: 10px 0;'>
            <strong>🔍 Tool Invoked:</strong> hotel_booking("{destination}")
        </div>
    """)
    )

    # Simulate availability check
    cities_with_rooms = ["stockholm", "seattle", "tokyo", "london", "amsterdam"]
    has_rooms = destination.lower() in cities_with_rooms

    result = {"has_availability": has_rooms, "destination": destination}

    return json.dumps(result)


print("✅ hotel_booking tool created with @ai_function decorator")

✅ hotel_booking tool created with @ai_function decorator


## Step 3: మార్గ నిర్ధారణ కోసం షరతుల ఫంక్షన్స్ నిర్వచించండి

ఈ ఫంక్షన్లు ఏజెంట్ యొక్క స్పందనను పరిశీలించి, వర్క్‌ఫ్లోలో ఏ దారిని అనుసరించాలి అనే విషయాన్ని నిర్ణయిస్తాయి.

**ప్రధాన విధానం:**
1. సందేశం `AgentExecutorResponse` కావాలని చూడండి
2. నిర్మాణాత్మక అవుట్‌పుట్ (Pydantic మోడల్) ను పార్స్ చేయండి
3. మార్గ నిర్ణయానికి `True` లేదా `False` ను రిటర్న్ చేయండి

వర్క్‌ఫ్లో ఈ షరతులను **ఎడ్జిలపై** బదులుగా ఏ కార్యనిర్వాహకుడు (executor) ను తర్వాత పిలవాలో నిర్ణయించడానికి ఉపయోగిస్తుంది.


In [4]:
def has_availability_condition(message: Any) -> bool:
    """
    Condition for routing when hotels ARE available.
    
    Returns True if the destination has hotel rooms.
    """
    if not isinstance(message, AgentExecutorResponse):
        return True  # Default to True if unexpected type

    try:
        result = BookingCheckResult.model_validate_json(message.agent_run_response.text)

        display(
            HTML(f"""
            <div style='padding: 12px; background: #c8e6c9; border-left: 4px solid #4caf50; border-radius: 4px; margin: 10px 0;'>
                <strong>✅ Condition Check:</strong> has_availability = <strong>{result.has_availability}</strong> for {result.destination}
            </div>
        """)
        )

        return result.has_availability
    except Exception as e:
        display(
            HTML(f"""
            <div style='padding: 12px; background: #ffcdd2; border-left: 4px solid #f44336; border-radius: 4px; margin: 10px 0;'>
                <strong>⚠️  Error:</strong> {str(e)}
            </div>
        """)
        )
        return False


def no_availability_condition(message: Any) -> bool:
    """
    Condition for routing when hotels are NOT available.
    
    Returns True if the destination has no hotel rooms.
    """
    if not isinstance(message, AgentExecutorResponse):
        return False

    try:
        result = BookingCheckResult.model_validate_json(message.agent_run_response.text)

        display(
            HTML(f"""
            <div style='padding: 12px; background: #ffecb3; border-left: 4px solid #ff9800; border-radius: 4px; margin: 10px 0;'>
                <strong>❌ Condition Check:</strong> no_availability for {result.destination}
            </div>
        """)
        )

        return not result.has_availability
    except Exception as e:
        return False


print("✅ Condition functions defined:")
print("   - has_availability_condition (routes when rooms exist)")
print("   - no_availability_condition (routes when no rooms)")

✅ Condition functions defined:
   - has_availability_condition (routes when rooms exist)
   - no_availability_condition (routes when no rooms)


## దశ 4: కస్టమ్ డిస్ప్లే ఎగ్జిక్యూటర్ సృష్టించండి

ఎగ్జిక్యూటర్స్ అనేవి వర్క్‌ఫ్లో భాగాలు, అవి ట్రాన్స్‌ఫర్మేషన్లు లేదా పక్క ప్రভাবాలు చేస్తాయి. చివరి ఫలితాన్ని ప్రదర్శించడానికి కస్టమ్ ఎగ్జిక్యూటర్ సృష్టించడానికి `@executor` డెకొరేటరును ఉపయోగిస్తాము.

**ముఖ్య భావనలు:**
- `@executor(id="...")` - ఫంక్షన్‌ను వర్క్‌ఫ్లో ఎగ్జిక్యూటర్‌గా నమోదు చేస్తుంది
- `WorkflowContext[Never, str]` - ఇన్‌పుట్/అవుట్పుట్ కోసం టైప్ సంకేతాలు
- `ctx.yield_output(...)` - చివరి వర్క్‌ఫ్లో ఫలితాన్ని ఇస్తుంది


In [5]:
@executor(id="display_result")
async def display_result(response: AgentExecutorResponse, ctx: WorkflowContext[Never, str]) -> None:
    """
    Display the final result as workflow output.
    
    This executor receives the final agent response and yields it as the workflow output.
    """
    display(
        HTML("""
        <div style='padding: 15px; background: #f3e5f5; border-left: 4px solid #9c27b0; border-radius: 4px; margin: 10px 0;'>
            <strong>📤 Display Executor:</strong> Yielding workflow output
        </div>
    """)
    )

    await ctx.yield_output(response.agent_run_response.text)


print("✅ display_result executor created with @executor decorator")

✅ display_result executor created with @executor decorator


## Step 5: పర్యావరణ సూచికలను లోడ్ చేయండి

LLM క్లయింట్‌ను అందంగా అమర్చండి. ఈ ఉదాహరణ క్రింది వాటితో పని చేస్తుంది:
- **GitHub మోడల్స్** (GitHub టోకెన్‌తో ఉచిత స్థాయి)
- **Azure OpenAI**
- **OpenAI**


In [6]:
# Load environment variables
load_dotenv()

# Check for GitHub Models or OpenAI
chat_client = OpenAIChatClient(base_url=os.environ.get(
    "GITHUB_ENDPOINT"), api_key=os.environ.get("GITHUB_TOKEN"), model_id="gpt-4o")

## దశ 6: నిర్మాణాత్మక అవుట్‌పుట్‌లతో AI ఏజెంట్లను సృష్టించండి

మేము **మూడు ప్రత్యేక ఏజెంట్లను** సృష్టిస్తాము, ప్రతి ఒక్కటిని `AgentExecutor` లో ముడిపెడతాము:

1. **availability_agent** - టూల్ ఉపయోగించి హోటల్ అందుబాటును తనిచేస్తుంది  
2. **alternative_agent** - ఇతర నగరాలను సూచిస్తుంది (రూం లేనిప్పుడు)  
3. **booking_agent** - బుకింగ్కు ప్రోత్సహిస్తుంది (రూం లభ్యమైతే)

**ప్రధాన లక్షణాలు:**  
- `tools=[hotel_booking]` - ఏజెంట్కు టూల్‌ను అందిస్తుంది  
- `response_format=PydanticModel` - నిర్మాణాత్మక JSON అవుట్‌పుట్‌ను బలవంతం చేస్తుంది  
- `AgentExecutor(..., id="...")` - వర్క్‌ఫ్లో ఉపయోగానికిగానూ ఏజెంట్ను ముడిపెడుతుంది


In [7]:
# Agent 1: Check availability with tool
availability_agent = AgentExecutor(
    chat_client.create_agent(
        instructions=(
            "You are a hotel booking assistant that checks room availability. "
            "Use the hotel_booking tool to check if rooms are available at the destination. "
            "Return JSON with fields: destination (string), has_availability (bool), and message (string). "
            "The message should summarize the availability status."
        ),
        tools=[hotel_booking],
        response_format=BookingCheckResult,
    ),
    id="availability_agent",
)

# Agent 2: Suggest alternative (when no rooms)
alternative_agent = AgentExecutor(
    chat_client.create_agent(
        instructions=(
            "You are a helpful travel assistant. When a user cannot find hotels in their requested city, "
            "suggest an alternative nearby city that has availability. "
            "Return JSON with fields: alternative_destination (string) and reason (string). "
            "Make your suggestion sound appealing and helpful."
        ),
        response_format=AlternativeResult,
    ),
    id="alternative_agent",
)

# Agent 3: Suggest booking (when rooms available)
booking_agent = AgentExecutor(
    chat_client.create_agent(
        instructions=(
            "You are a booking assistant. The user has found available hotel rooms. "
            "Encourage them to book by highlighting the destination's appeal. "
            "Return JSON with fields: destination (string), action (string), and message (string). "
            "The action should be 'book_now' and message should be encouraging."
        ),
        response_format=BookingConfirmation,
    ),
    id="booking_agent",
)

display(
    HTML("""
    <div style='padding: 15px; background: #e3f2fd; border-left: 4px solid #2196f3; border-radius: 4px; margin: 10px 0;'>
        <strong>✅ Created 3 Agents:</strong>
        <ul style='margin: 10px 0 0 0;'>
            <li><strong>availability_agent</strong> - Checks availability with hotel_booking tool</li>
            <li><strong>alternative_agent</strong> - Suggests alternative cities</li>
            <li><strong>booking_agent</strong> - Encourages booking</li>
        </ul>
    </div>
""")
)

## దశ 7: కండీషనల్ ఎడ్జులతో వర్క్‌ఫ్లో ని నిర్మించడం

ఇప్పుడు కండీషనల్ రూటింగ్ తో గ్రాఫ్ నిర్మించడానికి `WorkflowBuilder` ని ఉపయోగిస్తాం:

**వర్క్‌ఫ్లో నిర్మాణం:**
```
availability_agent (START)
        ↓
   Evaluate conditions
        ↙         ↘
[no_availability]  [has_availability]
        ↓              ↓
alternative_agent  booking_agent
        ↓              ↓
    display_result ←───┘
```

**ప్రధాన మెథడ్లు:**
- `.set_start_executor(...)` - ఎంట్రీ పాయింట్ ను సెట్ చేస్తుంది
- `.add_edge(from, to, condition=...)` - కండీషనల్ ఎడ్జ్ ను జోడిస్తుంది
- `.build()` - వర్క్‌ఫ్లోను తుది రూపంలో మార్చుతుంది


In [8]:
# Build the workflow with conditional routing
workflow = (
    WorkflowBuilder()
    .set_start_executor(availability_agent)
    # NO AVAILABILITY PATH
    .add_edge(availability_agent, alternative_agent, condition=no_availability_condition)
    .add_edge(alternative_agent, display_result)
    # HAS AVAILABILITY PATH
    .add_edge(availability_agent, booking_agent, condition=has_availability_condition)
    .add_edge(booking_agent, display_result)
    .build()
)

display(
    HTML("""
    <div style='padding: 20px; background: linear-gradient(135deg, #667eea 0%, #764ba2 100%); color: white; border-radius: 8px; margin: 10px 0;'>
        <h3 style='margin: 0 0 15px 0;'>✅ Workflow Built Successfully!</h3>
        <p style='margin: 0; line-height: 1.6;'>
            <strong>Conditional Routing:</strong><br>
            • If <strong>NO availability</strong> → alternative_agent → display_result<br>
            • If <strong>availability</strong> → booking_agent → display_result
        </p>
    </div>
""")
)

## Step 8: టెస్ట్ కేసు 1 నడపండి - అందుబాటులో లేకపోయిన నగరం (పారిస్)

మన సిమ్యులేషన్‌లో గది లేని పారిస్ నగరంలో హోటల్స్ అభ్యర్థించటం ద్వారా **అందుబాటులో లేకపోవడం** మార్గాన్ని పరీక్షిద్దాం.


In [9]:
display(
    HTML("""
    <div style='padding: 20px; background: #fff3e0; border-left: 4px solid #ff9800; border-radius: 8px; margin: 20px 0;'>
        <h3 style='margin: 0 0 10px 0; color: #e65100;'>🧪 TEST CASE 1: Paris (No Availability)</h3>
        <p style='margin: 0;'>Expected workflow path: availability_agent → alternative_agent → display_result</p>
    </div>
""")
)

# Create request for Paris
request_paris = AgentExecutorRequest(
    messages=[ChatMessage(Role.USER, text="I want to book a hotel in Paris")], should_respond=True
)

# Run the workflow
events_paris = await workflow.run(request_paris)
outputs_paris = events_paris.get_outputs()

# Display results
if outputs_paris:
    result_paris = AlternativeResult.model_validate_json(outputs_paris[0])

    display(
        HTML(f"""
        <div style='padding: 25px; background: linear-gradient(135deg, #FFD700 0%, #FFA500 100%); border-radius: 12px; box-shadow: 0 4px 12px rgba(255,165,0,0.3); margin: 20px 0;'>
            <h3 style='margin: 0 0 15px 0; color: #333;'>🏆 WORKFLOW RESULT (Paris)</h3>
            <div style='background: white; padding: 20px; border-radius: 8px;'>
                <p style='margin: 0 0 10px 0; font-size: 16px;'><strong>Status:</strong> ❌ No rooms in Paris</p>
                <p style='margin: 0 0 10px 0; font-size: 16px;'><strong>Alternative Suggestion:</strong> 🏨 {result_paris.alternative_destination}</p>
                <p style='margin: 0; font-size: 14px; color: #666;'><strong>Reason:</strong> {result_paris.reason}</p>
            </div>
        </div>
    """)
    )

## Step 9: రన్ టెస్ట్ కేసు 2 - నగరం WITH అందుబాటు (స్టాక్‌హోమ్)

ఇప్పుడు మనము **అందుబాటు** మార్గాన్ని పరీక్షిద్దాం స్టాక్‌హోమ్ లో హోటల్స్ అభ్యర్థిస్తూ (మా సిమ్యులేషన్ లో గదులు ఉన్నవి).


In [10]:
display(
    HTML("""
    <div style='padding: 20px; background: #e8f5e9; border-left: 4px solid #4caf50; border-radius: 8px; margin: 20px 0;'>
        <h3 style='margin: 0 0 10px 0; color: #1b5e20;'>🧪 TEST CASE 2: Stockholm (Has Availability)</h3>
        <p style='margin: 0;'>Expected workflow path: availability_agent → booking_agent → display_result</p>
    </div>
""")
)

# Create request for Stockholm
request_stockholm = AgentExecutorRequest(
    messages=[ChatMessage(Role.USER, text="I want to book a hotel in Stockholm")], should_respond=True
)

# Run the workflow
events_stockholm = await workflow.run(request_stockholm)
outputs_stockholm = events_stockholm.get_outputs()

# Display results
if outputs_stockholm:
    result_stockholm = BookingConfirmation.model_validate_json(outputs_stockholm[0])

    display(
        HTML(f"""
        <div style='padding: 25px; background: linear-gradient(135deg, #4caf50 0%, #8bc34a 100%); color: white; border-radius: 12px; box-shadow: 0 4px 12px rgba(76,175,80,0.3); margin: 20px 0;'>
            <h3 style='margin: 0 0 15px 0;'>🏆 WORKFLOW RESULT (Stockholm)</h3>
            <div style='background: white; color: #333; padding: 20px; border-radius: 8px;'>
                <p style='margin: 0 0 10px 0; font-size: 16px;'><strong>Status:</strong> ✅ Rooms Available!</p>
                <p style='margin: 0 0 10px 0; font-size: 16px;'><strong>Destination:</strong> 🏨 {result_stockholm.destination}</p>
                <p style='margin: 0 0 10px 0; font-size: 16px;'><strong>Action:</strong> {result_stockholm.action}</p>
                <p style='margin: 0; font-size: 14px; color: #666;'><strong>Message:</strong> {result_stockholm.message}</p>
            </div>
        </div>
    """)
    )

## ప్రధాన అంశాలు మరియు తదుపరి దశలు

### ✅ మీరు నేర్చుకున్నవీ:

1. **WorkflowBuilder నమూనా**
   - ప్రవేశ దశను నిర్వచించడానికి `.set_start_executor()` ఉపయోగించండి
   - షరతు ఆధారిత మార్గనిర్దేశానికి `.add_edge(from, to, condition=...)` ఉపయోగించండి
   - వర్క్‌ఫ్లోని తుది రూపంలోకి తేవడానికి `.build()` ను పిలవండి

2. **షరతు ఆధారిత మార్గనిర్దేశం**
   - కండిషన్ ఫంక్షన్లు `AgentExecutorResponse` ని పరిశీలిస్తాయి
   - మార్గనిర్దేశ నిర్ణయాలు తీసుకోవడానికి నిర్మిత అవుట్పుట్‌లను పార్స్ చేస్తాయి
   - ఒక ఎడ్జ్‌ను యాక్టివేట్ చేయడానికి `True` తిరిగి ఇస్తాయి, దాటవేయడానికి `False`

3. **టూల్ ఇంటిగ్రేషన్**
   - Python ఫంక్షన్లను AI టూల్స్ గా మార్చడానికి `@ai_function` ఉపయోగించండి
   - అవసరం ఉన్నప్పుడు ఏజెంట్లు టూల్స్‌ను ఆటోమేటిక్‌గా పిలుస్తాయి
   - టూల్స్ JSON ను తిరిగి ఇస్తాయి, ఏజెంట్లు వాటిని పార్స్ చేయగలవు

4. **నిర్మిత అవుట్పుట్లు**
   - టైపు-సేఫ్ డేటా ఎక్స్ట్రాక్షన్ కోసం Pydantic మోడల్స్ ఉపయోగించండి
   - ఏజెంట్లు సృష్టించే సమయంలో `response_format=MyModel` సెట్ చేయండి
   - ప్రతిస్పందనలను `Model.model_validate_json()` తో పార్స్ చేయండి

5. **కస్టమ్ ఎగ్జిక్యూటర్లు**
   - వర్క్‌ఫ్లో భాగాలను సృష్టించడానికి `@executor(id="...")` ఉపయోగించండి
   - ఎగ్జిక్యూటర్లు డేటాను మార్చగలవు లేదా పక్క ఫలితాలు చేయగలవు
   - వర్క్‌ఫ్లో ఫలితాలను ఉత్పత్తి చేయడానికి `ctx.yield_output()` ఉపయోగించండి

### 🚀 వాస్తవ ప్రపంచ అనువర్తనాలు:

- **ప్రయాణ బుకింగ్**: అందుబాటును తనిఖీ చేయండి, ప్రత్యామ్నాయాలను సూచించండి, ఎంపికలను సరిపోల్చండి
- **కస్టమర్ సర్వీస్**: సమస్య రకం, భావోద్వేగం, ప్రాధాన్యత ఆధారంగా మార్గనిర్దేశం
- **ఈ-కామర్స్**: స్టాక్ తనిఖీ, ప్రత్యామ్నాయాలను సూచించడం, ఆదేశాలను ప్రాసెస్ చేయడం
- **కంటెంట్ మోడరేషన్**: టాక్సిసిటీ స్కోర్లు, యూజర్ ఫ్లాగుల ఆధారంగా మార్గనిర్దేశం
- **అనుమతి వర్క్‌ఫ్లోలు**: మొత్తం, యూజర్ పాత్ర, ప్రమాద స్థాయి ఆధారంగా మార్గనిర్దేశం
- **బహుళ దశా ప్రాసెసింగ్**: డేటా నాణ్యత, పూర్తి స్థితి ఆధారంగా మార్గనిర్దేశం

### 📚 తదుపరి దశలు:

- మరింత సంక్లిష్టమైన షరతులను (అनेक క్రీటీరియా) జోడించండి
- వర్క్‌ఫ్లో స్థితి నిర్వహణతో లూప్‌లను అమలు చేయండి
- పునర్వినియోగ సంబంధిత భాగాలకు ఉప-వర్క్‌ఫ్లోలను జోడించండి
- వాస్తవ APIలతో (హోటల్ బుకింగ్, స్టాక్ సిస్టమ్స్) ఇంటిగ్రేట్ చేయండి
- లోప నిర్వహణ మరియు బ్యాకప్ మార్గాలు జోడించండి
- బిల్ట్-ఇన్ విజువలైజేషన్ టూల్స్ తో వర్క్‌ఫ్లోలను విజువలైజ్ చేయండి


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**డిస్క్లైమర్**:  
ఈ పత్రం AI అనువాద సేవ [Co-op Translator](https://github.com/Azure/co-op-translator) ఉపయోగించి అనువదించబడింది. మేము నిశ్చితత్వానికి ప్రయత్నిస్తున్నప్పటికీ, ఆటోమేటెడ్ అనువాదాల్లో పొరపాట్లు లేదా అసమతుల్యతలు ఉండవచ్చు. మాతృభాషలో ఉన్న అసలు పత్రం అధికారిక మాధ్యమంగా పరిగణించాలి. ముఖ్యమైన సమాచారం కోసం, ప్రొఫెషనల్ మానవ అనువాదం చేయించుకోవడం ఉత్తమం. ఈ అనువాదం వాడకం వలన కలిగే ఏదైనా అపవ్యతిరేకతలకు మేము బాధ్యత వహించము.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
